## Insertion of Fact Table

In [0]:
%sql
USE CATALOG olist_ecommerce_project;

### Importing Libraries

In [0]:
from pyspark.sql.functions import ( 
                                   sum as spark_sum, count , when, col,
                                   coalesce, to_date, year, month, dayofmonth, hour
)
from pyspark.sql.types import DateType

#### Creating Fact table

##### Step 1: Start with Silver Orders (base table)

In [0]:
df_orders = spark.table("olist_ecommerce_project.silver.slv_orders")

print("Orders base table rows:", df_orders.count())
df_orders.select("order_id", "order_status", "order_purchase_timestamp").show(3, truncate=False)

#####  Step 2: Aggregate Order Items to order level

In [0]:
df_order_items = spark.table("olist_ecommerce_project.silver.slv_order_items")

# Aggregate items by order_id
df_items_agg = (
    df_order_items
    .groupBy("order_id")
    .agg(
        count("*").alias("total_items_count"),
        spark_sum("product_id").alias("dummy_sum")  # Just to get distinct products below
    )
)

# Count distinct products per order
df_items_agg = (
    df_order_items
    .groupBy("order_id")
    .agg(
        count("*").alias("total_items_count"),
        count(when(col("product_id").isNotNull(), 1)).alias("distinct_products_count"),
        spark_sum("price").alias("total_price"),
        spark_sum("freight_value").alias("total_freight_value"),
        spark_sum("total_item_value").alias("total_order_value")
    )
)

print("Items aggregated by order:", df_items_agg.count())
df_items_agg.show(3, truncate=False)

In [0]:
df_order_items.show(5, truncate=False)

#####  Step 3: Get Payments (already aggregated at Silver level)

In [0]:
df_payments = spark.table("olist_ecommerce_project.silver.slv_payments")

print("Payments aggregated rows:", df_payments.count())
df_payments.select("order_id", "total_payment_value", "payment_methods_used").show(3, truncate=False)

#####  Step 4: Get Reviews (already deduplicated at Silver level)

In [0]:
df_reviews = spark.table("olist_ecommerce_project.silver.slv_reviews")

df_reviews_select = (
    df_reviews
    .select(
        "order_id",
        "review_score",
        "sentiment_category",
        "has_comment"
    )
)

print("Reviews rows:", df_reviews_select.count())
df_reviews_select.show(3, truncate=False)

#####  Step 5: Join everything together

In [0]:

# Start with orders
df_fact = df_orders

# Join items (left join — some orders might have no items recorded)
df_fact = df_fact.join(
    df_items_agg,
    on="order_id",
    how="left"
)

# Join payments (left join — some orders might have no payments)
df_fact = df_fact.join(
    df_payments,
    on="order_id",
    how="left"
)

# Join reviews (left join — not all orders have reviews)
df_fact = df_fact.join(
    df_reviews_select,
    on="order_id",
    how="left"
)

# Add date_key for join with dim_date
df_fact = df_fact.withColumn(
    "date_key",
    (year("order_purchase_timestamp") * 10000 +
     month("order_purchase_timestamp") * 100 +
     dayofmonth("order_purchase_timestamp")).cast("integer")
)

print("Fact orders rows:", df_fact.count())
df_fact.select(
    "order_id",
    "order_status",
    "total_items_count",
    "total_order_value",
    "total_payment_value",
    "review_score",
    "date_key"
).show(5, truncate=False)

In [0]:
# Check all available columns after joins
print("Available columns in df_fact:")
print(df_fact.columns)

In [0]:
df

##### Step 6: Final cleanup and write

In [0]:
# Select only columns that actually exist in df_fact
df_fact_final = df_fact.select(
    "order_id",
    "customer_id",
    "order_status",
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
    "delivery_delay_days",
    "is_late",
    "is_delivery_date_missing",
    "total_items_count",
    "distinct_products_count",
    "total_price",
    "total_freight_value",
    "total_order_value",
    "total_payment_value",
    "payment_types",
    "payment_methods_used",
    "max_payment_installments",
    "review_score",
    "sentiment_category",
    "has_comment",
    "date_key"
)

print("Final fact table rows:", df_fact_final.count())


# Write to Gold
(
    df_fact_final.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("olist_ecommerce_project.gold.fact_orders")
)

print("fact_orders written successfully")